<a href="https://colab.research.google.com/github/AlHartMos/IEU_courses/blob/main/principals_of_programming/PP_data_structures_dictionaries_advanced_class.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Lecture on Dictionaries

This notebook is an **in-depth** guide to Python dictionaries.

Dictionaries are one of the most important tools in programming:
- they give you fast lookup by **key**
- they enable classic patterns: **counting**, **grouping**, **indexing**, **memoization**, **inverting mappings**
- they are implemented (conceptually) as **hash tables**


---

## Notebook structure
1. Recap: what is a dict + hash-table idea + why useful + performance table  
2. Creating dictionaries (literals, loops, comprehensions)  
3. Reading/accessing data (`[]`, `get`, `.keys()/.values()/.items()`) — views vs lists  
4. Updating / deleting / removing  
5. Membership checks (`in`)  
6. Iteration patterns  
7. Common dict methods  
8. Mutability model (hashable keys, mutable values, aliasing)  
9. Core CS ideas (hashing, collisions, probing, resizing)  
10. Main usage patterns (counting, grouping, indexing, inversion and memorization)  
11. Nested dicts  
12. Shallow vs deep copies  
13. Tuples as composite keys  
14. Advanced challenges

---

✅ **Recommended workflow:** run each code cell, tweak examples, and test your understanding.


In [ ]:
print('Ready ✅')

## 1) Recap — What is a dictionary?

A **dictionary** (type `dict`) stores a mapping:

$$\text{key} \rightarrow \text{value}$$

You give a key, Python returns the associated value.

```python
d = {"Ada": 10, "Bob": 7}
print(d["Ada"])  # 10
```

### Why dictionaries are useful
Dictionaries are great when you need:
- **fast membership**: “Have I seen this key before?”
- **fast lookup**: “What value is associated with this key?”
- **fast updates**: “Increment a counter / update a record”

They are a core tool for:
- frequency tables
- grouping data
- indexing (building fast lookup tables)
- graph representations (adjacency maps)
- caching results

---

## 1.1 The key CS idea: dictionaries are hash tables (conceptually)

A dictionary uses a **hash function**:

```python
h = hash(key)
```

to convert a key into an integer. Then it uses that integer to choose a slot in an internal table.

Think: **a cabinet with many drawers**.  
The hash tells you which drawer to start with.

- If the drawer is empty → store the `(key, value)` there
- If the drawer has the same key → update value
- If the drawer has a different key → collision → try another drawer (probing)


## 1.2 A simple hashing function for strings

A good hash function should:
- be **fast**
- mix the characters so that small changes in the key produce very different hashes
- reduce **collisions** (different keys mapping to the same hash)

A common *realistic* idea is a **polynomial rolling hash**:

$$h = (((c_0)\cdot p + c_1)\cdot p + c_2)\cdot p + \dots$$

expands to:

$$h = c_0\,p^{\,n-1} + c_1\,p^{\,n-2} + \cdots + c_{n-2}\,p + c_{n-1}$$


where:
- \(c_i\) is the numeric code of the i-th character (e.g., ASCII/Unicode code)
- \(p\) is a small base (often 31, 37, 53, …)


#### Example with `p = 31`
Let’s hash `"cat"`.

Character codes:
- `c` = 99  
- `a` = 97  
- `t` = 116  

Compute step by step:

$$h_0 = 0$$
$$h_1 = h_0 \cdot 31 + 99 = 99$$
$$h_2 = h_1 \cdot 31 + 97 = 99\cdot 31 + 97 = 3166$$
$$h_3 = h_2 \cdot 31 + 116 = 3166\cdot 31 + 116 = 98262$$

So the hash is \(h = 98262\).

If the table has \(N = 10\) slots:

$$\text{index} = h \bmod N = 98262 \bmod 10 = 2$$


#### Why this has fewer collisions than for example the “sum of letters”
- The **order matters**:
  - `"cat"` and `"tac"` will not hash to the same value.
- Characters contribute differently depending on position (because of the multiplication by 31).

This is still not “cryptographic”, but it is much closer to the kinds of mixing ideas real hash tables use.


## 1.2 Performance: dict vs list (rule-of-thumb)

Below is a practical big-O comparison.  
(For dict operations, this is **average-case**.)



| Operation (same “task”)             | List command                               |                 Dict command |   List performance | Dict performance |
| ----------------------------------- | ------------------------------------------ | ---------------------------: | -----------------: | ---------------: |
| **Read one element**                | `lst[i]`                                   |                     `d[key]` |           **O(1)** |     **O(1) avg** |
| **Update one element**              | `lst[i] = x`                               |             `d[key] = value` |           **O(1)** |     **O(1) avg** |
| **Check presence**                  | `x in lst`                                 |                   `key in d` |           **O(n)** |     **O(1) avg** |
| **Grow the structure**              | `lst.append(x)`                            |             `d[key] = value` | **amortized O(1)** |     **O(1) avg** |
| **Bulk update / merge**             | `lst.extend(other)`  |            `d.update(other)` |           **O(m)** |     **O(m) avg** |
| **Insert one element (keep order)** | `lst.insert(i, x)`                         |     *(no positional insert)* |  **O(n)** (shifts) |            **—** |
| **Remove one element**              | `lst.pop(i)` or `del lst[i]`               | `d.pop(key)` or `del d[key]` |  **O(n)** (shifts) |     **O(1) avg** |

**Important nuance (dicts):** the “O(1)” assumes (1) good hashing and (2) not-too-many collisions. Also, hashing a large key can cost **O(length of key)** (e.g., long strings/tuples), and resizing is occasionally **O(n)** (rare, but happens).


### Exercise 1 — Choose the right structure
For each task, write **dict** or **list** and one sentence why.

1. Keep scores for students by name  
2. Store an ordered playlist where duplicates are allowed  
3. Count how many times each word appears in a text  
4. Check if a user ID has appeared before (many queries)  
5. Store the last 10 sensor readings in order  


In [ ]:
# ✍️ Your answers (as comments)
# 1. dict
# 2. list
# 3. dict
# 4. dict
# 5. list


## 2) Different ways to define / create a dictionary

### 2.1 Dictionary literal (most common)
```python
d = {"a": 1, "b": 2}
```

### 2.2 Build with a loop
Useful when you compute keys/values step-by-step.


In [1]:
squares = {}
for x in range(6):
    squares[x] = x * x
squares

{0: 0, 1: 1, 2: 4, 3: 9, 4: 16, 5: 25}

### 2.3 Dictionary comprehension
A compact way to build dicts:

```python
{key_expr: value_expr for item in iterable if condition}
```


In [2]:
squares2 = {x: x**2 for x in range(6)}
squares3 = {x: x**3 for x in range(6)}

print(squares2.values())
print(squares3.values())

dict_values([0, 1, 4, 9, 16, 25])
dict_values([0, 1, 8, 27, 64, 125])


### Exercise 2 — Create a dict from two lists
Given:

```python
names  = ["Ada", "Bob", "Carla"]
scores = [10, 7, 9]
```

Create a dict mapping name → score using:
1) a loop  
2) a dict comprehension  

(You may use `zip`.)


In [14]:
# Your turn:
names  = ["Ada", "Bob", "Carla"]
scores = [10, 7, 9]

name_scores = {names[i]: scores[i] for i in range(len(names))}
print(name_scores)

{'Ada': 10, 'Bob': 7, 'Carla': 9}


## 3) Reading and accessing data inside the dictionary

### 3.1 Access by key with `[]`
```python
d[key]
```
- fast
- raises `KeyError` if missing

### 3.2 Safe access with `.get`
```python
d.get(key, default)
```
- returns `default` (or `None`) if missing


In [4]:
d = {"Ada": 10, "Bob": 7}
print(d["Ada"])
print(d.get("Carla"))        # None
print(d.get("Carla", 10))     # default value


10
None
10


### 3.3 `.keys()`, `.values()`, `.items()`
- `d.keys()` → **view** of keys
- `d.values()` → **view** of values
- `d.items()` → **view** of `(key, value)` pairs

These return **views**, not lists:
- cheap to create (no copying)
- reflect changes in the dict


In [5]:
for key in d.keys():
    print(key)

Ada
Bob


In [6]:
list(d.keys())

['Ada', 'Bob']

In [7]:
d = {"a": 1, "b": 2}
ks = d.keys()
print("Before:", list(ks))

d["c"] = 3
print("After:", list(ks))  # view updates


Before: ['a', 'b']
After: ['a', 'b', 'c']


### Exercise 3 — Views vs lists
1) Create `ks = d.keys()`  
2) Create `ks_list = list(d.keys())`  
3) Add a new key to `d`  
4) Print `ks` and `ks_list` again and explain the difference.


In [10]:
d = {"x": 1, "y": 2}

# TODO
ks = d.keys()
ks_list = list(d.keys())
print("Before: ", ks, " ; ", ks_list)

d["z"] = 3
print("After: ", ks, " ; ", ks_list)


Before:  dict_keys(['x', 'y'])  ;  ['x', 'y']
After:  dict_keys(['x', 'y', 'z'])  ;  ['x', 'y']


## 4) Updating / deleting / removing

### 4.1 Insert / update
```python
d[key] = value
```

### 4.2 Delete with `del`
```python
del d[key]
```
Raises `KeyError` if missing.

### 4.3 Remove and return with `.pop`
```python
d.pop(key)         # KeyError if missing
d.pop(key, default)
```

### 4.4 Update many keys
```python
d.update(other_dict)
```


In [12]:
d1 = {"a": 1, "b": 2}
d2 = {"x": 9, "y": 10, "a":3}
d1.update(d2) # Adds d2 to d1, overwrites any duplicates
print(d1)

{'a': 3, 'b': 2, 'x': 9, 'y': 10}


In [13]:
d = {"a": 1, "b": 2}
d["c"] = 3          # insert
d["a"] = 100        # update
print(d)

removed = d.pop("b")
print("removed:", removed)
print(d)

d.update({"x": 9, "y": 10, "a":3})
print(d)

lst = [1,2,4]
del lst[0:2]
print(lst)

{'a': 100, 'b': 2, 'c': 3}
removed: 2
{'a': 100, 'c': 3}
{'a': 3, 'c': 3, 'x': 9, 'y': 10}
[4]


## 5) Membership checks

### Keys membership (fast on average)
```python
key in d
```
Checks keys only.

### Values membership (slow)
```python
value in d.values()
```
This is a linear scan (O(n)).


In [15]:
d = {"a": 1, "b": 2}
print("a" in d)         # True
print(2 in d)           # False (checks keys)
print(2 in d.values())  # True (but slower for large dicts)


True
False
True


### Exercise 4 — Robust delete
Write `safe_delete(d, key)` that:
- deletes the key if it exists
- does nothing otherwise
- returns True if deleted, False if not


In [16]:
def safe_delete(d, key):
    if key in d:
        del d[key]
        return True
    else:
        return False

d = {"Nacho": 10, "Ada":8}
print(safe_delete(d, "Maria"), d)

d.keys()

False {'Nacho': 10, 'Ada': 8}


dict_keys(['Nacho', 'Ada'])

## 6) Iteration over dictionaries

### 6.1 Iterate over keys (default)
```python
for k in d:
#    ...
```

### 6.2 Iterate over values
```python
for v in d.values():
    ...
```

### 6.3 Iterate over items (key/value pairs)
```python
for k, v in d.items():
    ...
```


In [17]:
d = {"Ada": 10, "Bob": 7, "Carla": 9}

print("Keys:")
for k in d:
    print(k)

print("Values:")
for v in d.values():
    print(v)

print("Items:")
for k, v in d.items():
    print(k, "->", v)


Keys:
Ada
Bob
Carla
Values:
10
7
9
Items:
Ada -> 10
Bob -> 7
Carla -> 9


## 7) Common dict methods

The methods below appear constantly in real code:

- `get(key, default)` — safe read
- `setdefault(key, default)` — ensure a key exists (useful for grouping)
- `pop(key, default)` — remove and return
- `update(other)` — merge/overwrite
- `clear()` — remove everything
- `copy()` — shallow copy

(You can inspect methods with `dir(d)` and read docs with `help(dict)`.)


### Exercise 5 — Group words by first letter
Given a list of words, build:
```python
{
  "a": ["ant", "apple"],
  "b": ["bee"],
  ...
}
```
Use `setdefault`.


In [42]:
# Your turn:
word_list = "item", "banana", "dragon", "robot", "cake", "camel"
word_dictionary = {}

for word in word_list:
  word_dictionary.setdefault(word[0], []).append(word)

print(word_dictionary)


{'i': ['item'], 'b': ['banana'], 'd': ['dragon'], 'r': ['robot'], 'c': ['cake', 'camel']}


## 8) Mutability model

### 8.1 Dicts are mutable
You can add/update/remove entries.

### 8.2 Keys must be hashable (usually immutable)

- `int`, `str`, `bool`, `float` ✅
- `tuple` ✅ (if all elements are hashable)
- `list`, `dict`, `set` ❌ (mutable → not hashable)

**Why**: If a list could be a key, its contents could change after insertion. A dict uses hash(key) to decide where to store the entry. If the list changes, its hash would change, and the dict would not be able to find the key reliably.

In [21]:
# Keys must be hashable:
d = {}
d[(2, 3)] = "tuple" # ?

In [22]:
d = {}
d[[2, 3]] = "list"  # ?

TypeError: unhashable type: 'list'


### 8.3 Values can be anything
Values can be mutable (lists, dicts, sets). This is powerful, but can lead to **aliasing** bugs.

#### Aliasing example (mutable values)
If two keys point to the same list, modifying it affects both.


In [23]:
shared = []
d = {"a": shared, "b": shared}

d["a"].append(1)
print(d)
d["b"].append(2)
print(d)

{'a': [1], 'b': [1]}
{'a': [1, 2], 'b': [1, 2]}


## 9) Main usage patterns: counting, grouping, indexing, inversion, memorization

### 9.1 Counting (frequency tables)

**Counting** means: you go through a sequence of items (words, characters, IDs, events, …) and build a dictionary that maps:

$$\text{item} \rightarrow \text{number of times it appears}$$

This produces a **frequency table**. Dictionaries are ideal for this because:
- each distinct item becomes a key
- updating the count is fast (average **O(1)** per item)

After building the frequency table, you can quickly answer questions like:
- “How many times does this item appear?”
- “What are the most common items?”
- “How many unique items are there?”


In [24]:
text = "to be or not to be"
freq = {}
for w in text.split():
    freq[w] = freq.get(w, 0) + 1
freq

{'to': 2, 'be': 2, 'or': 1, 'not': 1}

### 9.2 Grouping (collecting items by category)

**Grouping** means: you take many items and organize them by a **category key**, producing a dictionary that maps:

$$\text{category} \rightarrow \text{list/set of items in that category}$$

This is useful when multiple items belong to the same category (many-to-one relationship). Instead of storing a single value per key, each key stores a **collection** (often a list or set) of all items that share that category.

A common example is grouping students by a score label (e.g., `"high"`, `"mid"`, `"low"`), so you can quickly retrieve “all students in category X” without scanning the full list each time.



In [26]:
scores = {"Ada": 10, "Bob": 7, "Carla": 9, "Diego": 7}

groups = {}
for name, score in scores.items():
    label = "high" if score >= 9 else "mid" if score >= 7 else "low"
    groups.setdefault(label, []).append(name)

groups

{'high': ['Ada', 'Carla'], 'mid': ['Bob', 'Diego']}

### 9.3 Indexing (building a fast lookup table)

**Indexing** means: you take a collection of records (often a list of dictionaries or objects) and build a dictionary that maps:

$$\text{ID} \rightarrow \text{record}$$

The goal is to make lookups fast. Instead of scanning a list of records (which costs **O(n)**), you build an **index dictionary** once, so each lookup by ID becomes **O(1) on average**.

This is exactly the same idea as a database “index”: you pay a small cost upfront to build the index, then queries become much faster.



### Exercise 7 — Indexing student info with a dictionary to find student by ID


In [27]:
# Your turn:
students = [
    {"id": 101, "name": "Ada", "grade": 10},
    {"id": 102, "name": "Bob", "grade": 10},
    {"id": 103, "name": "Carla", "grade": 10},
]

### 9.4 Inversion (reverse mapping)

**Inversion** means: you start with a dictionary that maps:

$$\text{key} \rightarrow \text{value}$$

and you want a new structure that maps “the other way”:

$$\text{value} \rightarrow \text{key(s)}$$

Why “key(s)”? Because in the original dict, **many different keys can share the same value**.  
So when you invert, a single value often needs to map to **a collection of keys**.


In [28]:
def invert_multi(d):
    out = {}
    for k, v in d.items():
        out.setdefault(v, set()).add(k)
    return out

print(invert_multi({"a": 1, "b": 1, "c": 2}))


{1: {'a', 'b'}, 2: {'c'}}


### 9.5 Memoization (caching results)

**Memoization** means: *store the result of an expensive computation so you don’t recompute it.*

This is extremely common in:
- recursion (Fibonacci, dynamic programming)
- algorithms with overlapping subproblems
- speeding up repeated queries

Idea:
- Use a dictionary `cache` that maps **input → output**
- Before computing `f(x)`, check if `x` is already in the cache
  - if yes → return cached result
  - if no → compute, store, return

So a dict becomes a fast “memory” of results:
$$\text{cache}[x] = f(x)$$


In [ ]:
# ✅ Example: memoized Fibonacci using a dict
def fib(n, cache=None):
    if cache is None:
        cache = {0: 0, 1: 1}  # base cases

    if n in cache:           # fast lookup
        return cache[n]

    # compute once, store, reuse forever
    cache[n] = fib(n - 1, cache) + fib(n - 2, cache)
    return cache[n]

import time
start = time.perf_counter()
print(fib(35))   # fast because of caching
end = time.perf_counter()
print(f"Time:{(end - start)*1000:.4f} ms")


In [ ]:
def fib(n):
    # Base cases
    if n == 0:
        return 0
    if n == 1:
        return 1

    # Recursive definition (no memoization)
    return fib(n - 1) + fib(n - 2)

import time
start = time.perf_counter()
print(fib(35))   # fast because of caching
end = time.perf_counter()
print(f"Time:{(end - start)*1000:.4f} ms")

### Exercise 7 — Counting characters
Write a function `char_freq(s)` that returns a dict counting characters in a string.
Ignore spaces.


In [43]:
def char_freq(s):
    # TODO
    d = {}
    for char in s:
      d[char] = d.get(char, 0) + 1
    del d[" "]
    return d

print(char_freq("a b a"))


{'a': 2, 'b': 1}


## 11) Nested dictionaries

A nested dict is a dict whose values are also dicts.

Example:
```python
grades[student][course] = grade
```


In [ ]:
grades = {
    "Ada": {"Math": 9.5, "CS": 10.0},
    "Bob": {"Math": 6.0},
}

# Safe read:
print(grades.get("Carla", {}).get("CS"))


### Exercise 8 — Course → students inversion (nested dict)
Given `grades[student][course] = grade`, build `by_course[course][student] = grade`.


In [ ]:
grades = {
    "Ada": {"Math": 9.5, "CS": 10.0},
    "Bob": {"Math": 6.0, "CS": 7.5},
    "Carla": {"CS": 9.0},
}

# TODO


## 12) Shallow vs deep copies

### 12.1 Shallow copy
`d.copy()` creates a new dict, but **values are shared** if they are mutable.

### 12.2 Deep copy
A deep copy recursively copies nested structures (needs `copy.deepcopy`).


In [ ]:
import copy

d1 = {"a": [1, 2], "b": [3]}
d2 = d1.copy()          # shallow copy
d3 = copy.deepcopy(d1)  # deep copy

d2["a"].append(99)

print("d1:", d1)  # changed!
print("d2:", d2)
print("d3:", d3)  # unchanged


### Exercise 9 — Predict the output
Before running, predict what prints and explain why.

```python
d1 = {"x": [1]}
d2 = d1.copy()
d2["x"].append(2)
print(d1, d2)
```


In [ ]:
# ✍️ Your prediction/explanation:


## 13) Tuples as composite keys

Tuples are useful as keys when you need a key made of multiple parts:

- grid coordinates: `(row, col)`
- graph edges: `(u, v)`
- multi-parameter memoization: `(state1, state2, ...)`

Tuples are hashable **if** all their elements are hashable.


In [ ]:
grid = {}
grid[(0, 0)] = "start"
grid[(2, 3)] = "treasure"

print(grid[(2, 3)])


### Exercise 10 — Coin collection through a path on a grid using tuple keys

You are given two parallel lists: one with grid coordinates (tuples) and one with the number of coins at each coordinate:

```python
positions = [(0, 0), (0, 2), (1, 1), (2, 2), (3, 0)]
amounts   = [1,      3,      2,      5,      4]


1. Build a dictionary coins that maps each coordinate (row, col) to its coin amount.

2. Write a function coins_collected(coins, path) that takes a list of visited coordinates (a path) and returns the total coins collected along that path. Cordinate that are not in the dictionary should count as 0 coins.

In [ ]:
#Your turn:
positions = [(0, 0), (0, 2), (1, 1), (2, 2), (3, 0)]
amounts   = [1,      3,      2,      5,      4]

# Part A: build the dict

# Part B: compute coins collected along a path

## 14) Advanced challenges (with solutions)

These are longer problems that combine multiple dict concepts.
Try them as mini-projects.

---

### Challenge A — Anagram groups
Write `group_anagrams(words)` that groups words that are anagrams.

Example:
```python
group_anagrams(["eat","tea","tan","ate","nat","bat"])
```

Returns (order may vary):
```python
{
  ('a','e','t'): ["eat","tea","ate"],
  ('a','n','t'): ["tan","nat"],
  ('a','b','t'): ["bat"]
}
```

Requirements:
- Use a dict for grouping
- Use a **tuple** as the key, based on sorted letters


In [ ]:
# ✍️ Your turn
def group_anagrams(words):
    # TODO
    pass

# print(group_anagrams(["eat","tea","tan","ate","nat","bat"]))


---

### Challenge B — Build an adjacency dict (graph)
Given edges:

```python
edges = [("A","B"), ("A","C"), ("B","D"), ("E","F")]
```

Build:
```python
graph = {
  "A": {"B","C"},
  "B": {"A","D"},
  "C": {"A"},
  "D": {"B"},
  "E": {"F"},
  "F": {"E"}
}
```

Requirements:
- undirected graph: add both directions
- use `setdefault`


In [ ]:
edges = [("A","B"), ("A","C"), ("B","D"), ("E","F")]

# ✍️ Your turn
def build_graph(edges):
    # TODO
    pass

# print(build_graph(edges))


---

### Challenge C — Inverted index (tiny search engine)
Given docs:

```python
docs = {
  1: "to be or not to be",
  2: "to learn is to grow",
  3: "be the change you want to see"
}
```

Build an index:
```python
index[word] = set(doc_ids_that_contain_word)
```

Then write `and_query(index, words)` returning doc IDs containing *all* words.

Requirements:
- Use dict + sets
- Use set intersection


In [ ]:
docs = {
    1: "to be or not to be",
    2: "to learn is to grow",
    3: "be the change you want to see",
}

# ✍️ Your turn
def build_inverted_index(docs):
    # TODO
    pass

def and_query(index, words):
    # TODO
    pass

# idx = build_inverted_index(docs)
# print(idx)
# print(and_query(idx, {"to", "be"}))


---

## Summary / cheat-sheet

- Dict = key → value mapping  
- Keys must be **hashable** (usually immutable)  
- `key in d` checks keys (fast average)  
- `d.get(key, default)` avoids `KeyError`  
- `d.keys()/values()/items()` return **views** (live, not copied)  
- Use dicts for: counting, grouping, indexing, caching, graphs  

✅ If you can solve the challenges, you understand dicts at a strong CS level.
